# Contrastive Learning
While autoencoders are effective for learning embeddings in an unsupervised manner, newer **self-supervised techniques such as contrastive learning** have shown superior performance in preserving fine-grained similarities between nodes or graphs. 

Contrastive learning is a self-supervised framework designed to learn representations by **pulling semantically similar samples closer** and **pushing dissimilar samples apart** in the embedding space.  
Instead of reconstructing the input (as autoencoders do), contrastive learning shapes the geometry of the latent space directly, ensuring that similarity becomes an organizing principle for representation.

In graph representation learning, this approach has become central because graphs naturally contain positive and negative relational structure (e.g., connected vs. unconnected nodes, augmented views of the same graph, or subgraphs drawn from related regions).

---

## Core Idea

Given two paired inputs that should be similar—called a **positive pair**—the model must produce embeddings whose distance is small.  
Conversely, embeddings for **negative pairs** should be far apart.

Let  
- $x_i$ and $x_i'$ be a positive pair (two “views” of the same node or graph),  
- $x_i$ and $x_j'$ for $j \neq i$ be negative pairs.

An encoder $f_\theta(\cdot)$ produces representations:

$$
z_i = f_\theta(x_i), \qquad z_i' = f_\theta(x_i').
$$

The contrastive objective ensures:

- **alignment** of the positive pair,  
- **uniformity** of embeddings by spreading negative samples.

---

## Mathematical Formulation

### Similarity Function

A common choice is cosine similarity:

$$
\text{sim}(z, z') = \frac{z^\top z'}{\|z\|\|z'\|}.
$$

### InfoNCE Loss
> NCE: Noise-Contrastive Estimation


For a single sample $i$, the InfoNCE loss is defined as:

$$
\mathcal{L}_i = - \log \frac{\exp(\text{sim}(z_i, z_i') / \tau)}{\sum_{j=1}^{N} \exp(\text{sim}(z_i, z_j') / \tau)}
$$

where:

- $z_i$ = embedding of the anchor sample  
- $z_i'$ = embedding of the positive sample  
- $z_j'$ = embeddings of all candidate samples (including positives and negatives)  
- $\text{sim}(\cdot, \cdot)$ = similarity function (e.g., dot product or cosine)  
- $\tau$ = temperature parameter  

The full loss averaged over all samples is:

$$
\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} \mathcal{L}_i
$$

This objective encourages **high similarity for positives** and **low similarity for negatives**.

---

#### Gradient with Respect to Anchor Embedding

Assuming $\text{sim}(z_i, z_j') = z_i^\top z_j'$ (dot product), define the softmax probabilities:

$$
p_{ij} = \frac{\exp(z_i^\top z_j' / \tau)}{\sum_{k=1}^{N} \exp(z_i^\top z_k' / \tau)}
$$

Then the gradient of \(\mathcal{L}_i\) with respect to \(z_i\) is:

$$
\begin{aligned}
\frac{\partial \mathcal{L}_i}{\partial z_i} 
&= - \frac{\partial}{\partial z_i} \log \frac{e^{z_i^\top z_i' / \tau}}{\sum_j e^{z_i^\top z_j' / \tau}} \\
&= - \frac{1}{\tau} z_i' + \frac{1}{\tau} \sum_j p_{ij} z_j' \\
&= \frac{1}{\tau} \left( \sum_j p_{ij} z_j' - z_i' \right)
\end{aligned}
$$

#### Intuition:

- The positive sample $z_i'$ **pulls** the anchor $z_i$ closer.  
- The negatives, weighted by softmax probabilities $p_{ij}$, **push** $z_i$ away.  
- Gradients flow back through the encoder parameters $f_\theta(x_i)$, updating the embeddings indirectly.

---

## Contrastive Learning for Graphs

Graphs provide natural ways to define positive and negative pairs:

### 1. Node-Level Contrast (Local–Local)
Two augmented views of the same node’s neighborhood.

Used in: **DGI, GRACE, BGRL**

### 2. Subgraph-Level Contrast (Local–Global)
Contrast a node representation with a global summary of the entire graph.

Used in: **Deep Graph Infomax**

### 3. Graph-Level Contrast (Global–Global)
Two differently augmented versions of the full graph are positive pairs.

Used in: **GraphCL**

### 4. Cross-View Contrast
Contrast between views from different encoders or operators  
(e.g., message-passing vs. diffusion-based).

Used in: **MVGRL**

---

## Graph Augmentations

Graph-specific augmentations commonly include:

- node dropping  
- edge perturbation  
- feature masking  
- attribute corruption  
- random-walk subgraph sampling  
- diffusion-based smoothing  

Augmentations must preserve **semantic identity** while introducing variability.

---

## Comparison With Autoencoders

| Aspect | Autoencoder | Contrastive Learning |
|--------|-------------|---------------------|
| Objective | Reconstruct input | Maximize agreement between views |
| Embedding Shape | Indirect (via decoder) | Directly shaped by similarity |
| Negatives | Not required | Typically essential |
| Graph Augmentations | Optional | Crucial |
| Strength | Good for compression | Strong discriminative embeddings |
| Weakness | May collapse to identity mapping | Sensitive to augmentation quality |

Contrastive learning typically produces **higher-quality, task-effective** embeddings.

---

## Advantages in Graph Representation Learning

Contrastive learning is particularly effective for graphs due to:

- robustness to noise and sparsity  
- better capturing of **structural equivalence**  
- scalability  
- suitability for node/graph classification, clustering, and link prediction

End of this section